# Pipeline walkthrough

Runs each stage of the matcher on a bundled sample so you can inspect the
intermediate output rather than only the final score.

Run from the project root: `jupyter notebook notebooks/01_pipeline_walkthrough.ipynb`

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))

from src.config import SAMPLE_JOBS_DIR, SAMPLE_RESUMES_DIR
from src.parser import extract_text

resume_text = extract_text(SAMPLE_RESUMES_DIR / 'R006_fullstack_developer.txt')
job_text = extract_text(SAMPLE_JOBS_DIR / 'J002_backend_engineer.txt')
print(resume_text[:400])

## 1. Preprocessing

The original text is preserved alongside the processed form.

In [ ]:
from src.preprocessing import preprocess_text

processed = preprocess_text(resume_text)
print('words:', processed.word_count, '| sentences:', len(processed.sentences))
print('lemmas:', processed.lemmas[:20])

## 2. Skill extraction

Phrase matching, with the evidence line for each skill.

In [ ]:
from src.skill_extractor import extract_skills_detailed

found = extract_skills_detailed(resume_text)
for skill in found.skills[:12]:
    print(f"{skill:20} found as {found.surfaces_for(skill)}")

## 3. Job requirement analysis

In [ ]:
from src.job_analyzer import analyze_job_description

job = analyze_job_description(job_text)
print('title      :', job.title)
print('required   :', job.required_skills)
print('preferred  :', job.preferred_skills)
print('experience :', job.experience.describe())
print('education  :', job.education.describe())

## 4. Similarity: lexical vs semantic

The classic keyword-failure case: same meaning, almost no shared words.

In [ ]:
from src.similarity import calculate_semantic_similarity, calculate_tfidf_similarity

a = 'Developed RESTful APIs using Flask'
b = 'Experience building web APIs with Python frameworks'
print(f'TF-IDF   : {calculate_tfidf_similarity(a, b):.3f}')
print(f'Semantic : {calculate_semantic_similarity(a, b):.3f}')

## 5. Full pipeline

In [ ]:
from src.pipeline import match_resume_to_job

report = match_resume_to_job(resume_text, job_text)
print(report.overall_score, report.classification)
for reason in report.result.explanations:
    print(' +', reason)
for gap in report.result.gaps:
    print(' -', gap)

## 6. Score breakdown

`effective_weight` differs from `weight` when a component was unavailable and
its weight was redistributed.

In [ ]:
import pandas as pd

pd.DataFrame([
    {'component': c.label,
     'score': c.percent if c.available else None,
     'weight': c.weight,
     'effective': round(c.effective_weight, 3),
     'points': round(c.contribution, 1)}
    for c in report.result.components
])